# Diagnosing zero estimates

This notebook shows two reasons resource estimation can return no results.

## Set up resource estimation

Import the Q# and QRE APIs used by the existing samples.

In [1]:
import qdk
from qdk import qsharp
from qdk.qre import estimate
from qdk.qre.application import QSharpApplication
from qdk.qre.models import GateBased, RoundBasedFactory, SurfaceCode

## Define a small example application

A one-qubit T gate is enough to demonstrate both cases.

In [2]:
qsharp.eval("""
operation ZeroEstimateDemo() : Unit {
    use q = Qubit();
    T(q);
    MResetZ(q);
}
""")

app = QSharpApplication(qdk.code.ZeroEstimateDemo)
arch = GateBased(error_rate=1e-4, gate_time=100, measurement_time=500)

## Error budget

An extremely small `max_error` rejects every otherwise valid estimate. The warning reports the minimum error that was found.

In [3]:
estimate(
    app,
    arch,
    SurfaceCode.q(distance=3) * RoundBasedFactory.q(),
    max_error=1e-20,
)

Resource estimation produced zero results.
Run statistics: 32 trace(s), 104 ISA(s), 0 candidate job(s), and 0 successful estimate(s).
32 candidate estimate(s) exceeded the application's error budget.
The minimum error among candidates that otherwise estimated successfully was 0.000746000035. If max_error were greater than this value, at least one estimate would succeed.



[]

## Incompatible instruction set

The default trace transforms require both logical `T` states and lattice surgery. Keeping the T-state factory but omitting the surface-code transform leaves `LATTICE_SURGERY` unsupported.

In [4]:
estimate(
    app,
    arch,
    RoundBasedFactory.q(),
    max_error=0.01,
)

Resource estimation produced zero results.
Run statistics: 32 trace(s), 104 ISA(s), 0 candidate job(s), and 0 successful estimate(s).
No enumerated ISA provides all instructions required by the traces. Instructions absent from every ISA: LATTICE_SURGERY (4352). Apply additional or different trace transforms or ISA transforms to make the representations compatible.



[]